<a href="https://colab.research.google.com/github/matthew-ngzc/AI-Safety-Module/blob/main/Week_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone the repository

In [1]:
try:
    ! git clone https://github.com/NayMyatMin/CS427_SMU
    HOME_DIR = "./CS427_SMU/week5/"
except:
    print('Already clone!!!')

Cloning into 'CS427_SMU'...
remote: Enumerating objects: 415, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 415 (delta 4), reused 17 (delta 2), pack-reused 394 (from 1)
Receiving objects: 100% (415/415), 94.48 MiB | 16.06 MiB/s, done.
Resolving deltas: 100% (177/177), done.
Updating files: 100% (259/259), done.


# Exercise 1 (badnet sanitisation)
In this exercise, we will test whether transforming the sample at the inference time can effectively reduce the attack success rate of a badnet backdoor attack.

In [2]:
# @title
def shift_image_left(image, n):
    """
    Shifts the pixels of an image to the left by n pixels.
    Assumes image is a 2D or 3D NumPy array (H, W) or (H, W, C).
    Pixels shifted out from the left are filled with zeros (black).
    """
    if not isinstance(image, np.ndarray):
        image = np.array(image)

    # Handle 3D images (e.g., MNIST data with shape H, W, C or C, H, W)
    # Assuming format is (H, W, C) or (H, W) for simplicity, if C is first, needs adjustment.
    # For MNIST, it's typically (28, 28) or (1, 28, 28)

    shifted_image = np.zeros_like(image)

    if image.ndim == 2: # Grayscale image (H, W)
        shifted_image[:, :-n] = image[:, n:]
    elif image.ndim == 3: # Assuming (H, W, C)
        shifted_image[:, :-n, :] = image[:, n:, :]
    elif image.ndim == 4: # Assuming (N, H, W, C) or (N, C, H, W) for batch
        # This function is designed for a single image, so let's stick to 2D or 3D
        # If the input is (C, H, W), adjust accordingly:
        # shifted_image[:, :, :-n] = image[:, :, n:]
        # For typical MNIST, it's (H, W) for data[i] after ToTensor, it's (C, H, W)
        # If input is image.data[i] (H,W):
        shifted_image[:, :-n] = image[:, n:] # Assuming (H, W) input
    else:
        raise ValueError("Unsupported image dimensions. Expected 2D or 3D.")

    return shifted_image

In [3]:
# @title
def print_table(title, headers, rows):
    print("\n" + "="*50)
    print(title)
    print("="*50)

    # Print header
    header_line = "{:<20} {:<20} {:<15}".format(*headers)
    print(header_line)
    print("-"*50)

    # Print rows
    for row in rows:
        print("{:<20} {:<20} {:<15}".format(*row))


In [4]:
# @title
import torch
import os
import contextlib

from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
        self.fc4 = nn.Linear(10, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def load_model(model_class, name):
    model = model_class()
    model.load_state_dict(torch.load(name))

    return model


def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model.eval()
    loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

    loss /= num_batches
    accuracy = correct / size
    return accuracy, loss


device = 'cpu'
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

model = MNISTNet().to(device)
model = load_model(MNISTNet, HOME_DIR + 'exercise1/mnist2.pt')


# Suppress download messages
# with contextlib.redirect_stdout(open(os.devnull, 'w')), \
#      contextlib.redirect_stderr(open(os.devnull, 'w')):
#     train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
#     test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
# backdoor_test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# print('With original data')
# test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)
# test(model, test_loader, nn.CrossEntropyLoss(), device)

# for i in range(len(backdoor_test_dataset.data)):
#     backdoor_test_dataset.data[i][0][0] = 255
#     backdoor_test_dataset.data[i][0][1] = 255
#     backdoor_test_dataset.data[i][0][2] = 255
#     backdoor_test_dataset.data[i][1][0] = 255
#     backdoor_test_dataset.data[i][1][1] = 255
#     backdoor_test_dataset.data[i][1][2] = 255
#     backdoor_test_dataset.data[i][2][0] = 255
#     backdoor_test_dataset.data[i][2][1] = 255
#     backdoor_test_dataset.data[i][2][2] = 255

#     # TODO1: change the scale of the noise (30, 100, 500) and measure the attack success rate.
#     # noiseLevel = 10000
#     # noise = np.random.randint(-noiseLevel, noiseLevel, size=backdoor_test_dataset.data[i].shape)  # Random noise in range [-30, 30]
#     # backdoor_test_dataset.data[i] = np.clip(backdoor_test_dataset.data[i] + noise, 0, 255)  # Ensure valid pixel range

#     #TODO 2: comment out the noise above and instead write a few lines of code to shift the image to the right by 1, 5, 20 pixels
#     #and measure the attack success rate.
#     n_pixels_to_shift = 20 # You can change this value to 1, 5, or 20

#     # Convert the torch.ByteTensor to numpy array for the shifting function, then back to torch.ByteTensor
#     image_np = backdoor_test_dataset.data[i].numpy() # Convert to NumPy array
#     shifted_image_np = shift_image_left(image_np, n_pixels_to_shift)
#     backdoor_test_dataset.data[i] = torch.from_numpy(shifted_image_np).to(backdoor_test_dataset.data[i].dtype)

#     backdoor_test_dataset.targets[i] = 5

# print('With backdoored data')
# backdoor_test_loader = torch.utils.data.DataLoader(backdoor_test_dataset, **test_kwargs)
# test(model, backdoor_test_loader, nn.CrossEntropyLoss(), device)


TODO1: modify the noise level of the random noise added to see if it works for the following amounts of random noise:
- -30, 30
- -50, 50
- -500, 500 (it can go above the pixel value due to the clipping, which just makes it more likely to have a more extreme value even though the max pixel value is 255)

In [5]:
noise_levels = [30, 50, 500]
noise_results = []

for noiseLevel in noise_levels:

    # Reload fresh dataset each time
    #Suppress download messages
    with contextlib.redirect_stdout(open(os.devnull, 'w')), \
        contextlib.redirect_stderr(open(os.devnull, 'w')):
        backdoor_test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

    for i in range(len(backdoor_test_dataset.data)):
        # Add trigger
        backdoor_test_dataset.data[i][0:3, 0:3] = 255

        # Add noise
        noise = np.random.randint(-noiseLevel, noiseLevel,
                                  size=backdoor_test_dataset.data[i].shape)
        noisy_image = np.clip(
            backdoor_test_dataset.data[i].numpy() + noise,
            0, 255
        )

        backdoor_test_dataset.data[i] = torch.from_numpy(noisy_image).byte()

        # Set attack target
        backdoor_test_dataset.targets[i] = 5

    loader = DataLoader(backdoor_test_dataset, batch_size=1000)
    acc, loss = test(model, loader, nn.CrossEntropyLoss(), device)

    noise_results.append((
        f"(-{noiseLevel}, {noiseLevel})",
        f"{acc*100:.2f}%",
        f"{loss:.4f}"
    ))

TODO2: Shift and Pad the image and see if it works for the following left shifts:
- 1
- 5
- 20

In [6]:
shift_amounts = [1, 5, 20]
shift_results = []

for n_pixels_to_shift in shift_amounts:

    with contextlib.redirect_stdout(open(os.devnull, 'w')), \
        contextlib.redirect_stderr(open(os.devnull, 'w')):
        backdoor_test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

    for i in range(len(backdoor_test_dataset.data)):
        # Add trigger
        backdoor_test_dataset.data[i][0:3, 0:3] = 255

        # Shift image
        image_np = backdoor_test_dataset.data[i].numpy()
        shifted_image_np = shift_image_left(image_np, n_pixels_to_shift)

        backdoor_test_dataset.data[i] = torch.from_numpy(shifted_image_np).byte()

        # Set attack target
        backdoor_test_dataset.targets[i] = 5

    loader = DataLoader(backdoor_test_dataset, batch_size=1000)
    acc, loss = test(model, loader, nn.CrossEntropyLoss(), device)

    shift_results.append((
        str(n_pixels_to_shift),
        f"{acc*100:.2f}%",
        f"{loss:.4f}"
    ))


In [7]:
print_table(
    "TODO1: Adding Random Noise",
    ["Noise Level", "Attack Success Rate", "Avg Loss"],
    noise_results
)

print_table(
    "TODO2: Shifting and Padding",
    ["Pixels Shifted", "Attack Success Rate", "Avg Loss"],
    shift_results
)


TODO1: Adding Random Noise
Noise Level          Attack Success Rate  Avg Loss       
--------------------------------------------------
(-30, 30)            100.00%              0.0001         
(-50, 50)            100.00%              0.0001         
(-500, 500)          97.48%               0.0931         

TODO2: Shifting and Padding
Pixels Shifted       Attack Success Rate  Avg Loss       
--------------------------------------------------
1                    98.58%               0.0461         
5                    23.53%               3.9865         
20                   3.17%                2.5359         


1. Adding random noise:

Not so effective because the noise is not reliably changing the trigger pixels enough to remove the trigger activation. The model is still likely to recognise a bright spot in the top left corner


2. Shifting and Padding

Effective because the model learns positional information about the trigger, by shifting the pixels u destroy this link.



# Exercise 2 (retraining with model sanitisation)
In this exercise, we aim to evaluate whether retraining is effective in sanitizing the model (to get rid of the backdoor).

In [9]:
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import random

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
        self.fc4 = nn.Linear(10, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def save_model(model, name):
    torch.save(model.state_dict(), name)


def load_model(model_class, name, *args):
    model = model_class(*args)
    model.load_state_dict(torch.load(name, map_location=torch.device('cpu')))

    return model


def train(model, dataloader, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()

    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print('loss: {:.4f} [{}/{}]'.format(loss, current, size))


def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model.eval()
    loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

    loss /= num_batches
    accuracy = correct / size
    print('Test Result: Matching Expected Label @ {:.2f}%, Avg loss @ {:.4f}\n'.format(100 * accuracy, loss))
    return accuracy, loss


device = 'cpu'
train_kwargs = {'batch_size': 100}
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

model = load_model(MNISTNet, HOME_DIR + 'exercise2/badnet.pt')

test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, **test_kwargs)

results = []

print("Before retraining.")
print('With clean data')
clean_acc, clean_loss = test(model, test_loader, nn.CrossEntropyLoss(), device)

# Modify test data to test backdoor accuracy
backdoor_test_dataset = datasets.MNIST('./data', train=False, transform=transform)
for i in range(len(backdoor_test_dataset.data)):
    backdoor_test_dataset.data[i][0:3, 0:3] = 255
    # backdoor_test_dataset.data[i][0][0] = 255
    # backdoor_test_dataset.data[i][0][1] = 255
    # backdoor_test_dataset.data[i][0][2] = 255
    # backdoor_test_dataset.data[i][1][0] = 255
    # backdoor_test_dataset.data[i][1][1] = 255
    # backdoor_test_dataset.data[i][1][2] = 255
    # backdoor_test_dataset.data[i][2][0] = 255
    # backdoor_test_dataset.data[i][2][1] = 255
    # backdoor_test_dataset.data[i][2][2] = 255
    backdoor_test_dataset.targets[i] = 5

print('With backdoored data')
backdoor_test_loader = torch.utils.data.DataLoader(backdoor_test_dataset, **test_kwargs)
bd_acc, bd_loss = test(model, backdoor_test_loader, nn.CrossEntropyLoss(), device)

# store results
results.append(("Clean Data", f"{clean_acc*100:.2f}%", f"{clean_loss:.4f}"))
results.append(("Backdoored Data", f"{bd_acc*100:.2f}%", f"{bd_loss:.4f}"))

# Now we retrain
retrain_dataset = datasets.MNIST('./data', train=False, transform=transform)
#TODO: Vary the number of retraining samples from 10% (i.e., 1000) of the test dataset to 100% and observe whether
#retraining reduces the attack success rate.
retrain_samples = [1000, 5000, 10000]

for sample in retrain_samples:

    print(f"\n================ Retraining with {sample} clean samples ================\n")

    # Reload original backdoored model each time
    model = load_model(MNISTNet, HOME_DIR + 'exercise2/badnet.pt')

    retrain_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
    retrain_indexes = random.sample(range(len(retrain_dataset)), sample)
    retrain_subset = torch.utils.data.Subset(retrain_dataset, retrain_indexes)

    retrain_loader = DataLoader(retrain_subset, **train_kwargs)

    optimizer = optim.SGD(model.parameters(), lr=0.01)
    num_epochs = 20

    for epoch in range(num_epochs):
        print(f"\n------------- Epoch {epoch} -------------\n")
        train(model, retrain_loader, nn.CrossEntropyLoss(), optimizer, device)
        test(model, test_loader, nn.CrossEntropyLoss(), device)

    print("With backdoored data after retraining:")
    retrained_bd_acc, retrained_bd_loss = test(
        model,
        backdoor_test_loader,
        nn.CrossEntropyLoss(),
        device
    )

    results.append((f"Retrained ({sample})",
                    f"{retrained_bd_acc*100:.2f}%",
                    f"{retrained_bd_loss:.4f}"))


Before retraining.
With clean data
Test Result: Matching Expected Label @ 93.51%, Avg loss @ 0.2161

With backdoored data
Test Result: Matching Expected Label @ 89.57%, Avg loss @ 0.4193


================ Retraining with 1000 clean samples ================


------------- Epoch 0 -------------

loss: 0.3453 [0/1000]
Test Result: Matching Expected Label @ 93.99%, Avg loss @ 0.2025


------------- Epoch 1 -------------

loss: 0.3080 [0/1000]
Test Result: Matching Expected Label @ 94.10%, Avg loss @ 0.1994


------------- Epoch 2 -------------

loss: 0.2966 [0/1000]
Test Result: Matching Expected Label @ 94.14%, Avg loss @ 0.1979


------------- Epoch 3 -------------

loss: 0.2906 [0/1000]
Test Result: Matching Expected Label @ 94.22%, Avg loss @ 0.1967


------------- Epoch 4 -------------

loss: 0.2852 [0/1000]
Test Result: Matching Expected Label @ 94.22%, Avg loss @ 0.1958


------------- Epoch 5 -------------

loss: 0.2805 [0/1000]
Test Result: Matching Expected Label @ 94.31%, Avg 

In [10]:
print_table(
    "Retraining Backdoor Mitigation Results",
    ["Type", "Attack Success Rate", "Avg Loss"],
    results
)


Retraining Backdoor Mitigation Results
Type                 Attack Success Rate  Avg Loss       
--------------------------------------------------
Clean Data           93.51%               0.2161         
Backdoored Data      89.57%               0.4193         
Retrained (1000)     90.26%               0.4227         
Retrained (5000)     90.24%               0.3948         
Retrained (10000)    90.99%               0.3807         


We can see that it is not very effective, because catastrophic forgetting is not being triggered, indicated by the loss average loss. The distribution of the original data and the finetuning data would be around the same, so the model has no need to learn anything since it is already optimised for the same thing, so the parameters not changed and the backdoor stays.

# Extra
In this demo, we aim to show that it is rather easy to mitigate a backdoor if we know the trigger.

In [11]:
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import random

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
        self.fc4 = nn.Linear(10, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def save_model(model, name):
    torch.save(model.state_dict(), name)


def load_model(model_class, name, *args):
    model = model_class(*args)
    model.load_state_dict(torch.load(name, map_location=torch.device('cpu')))

    return model


def train(model, dataloader, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()

    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print('loss: {:.4f} [{}/{}]'.format(loss, current, size))


def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model.eval()
    loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

    loss /= num_batches
    correct /= size
    print('Test Result: Matching Expected Label @ {:.2f}%, Avg loss @ {:.4f}\n'.format(100 * correct, loss))


device = 'cpu'
train_kwargs = {'batch_size': 100}
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

model = load_model(MNISTNet, HOME_DIR + 'exercise2/badnet.pt')

test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)

print("Before retraining.")
print('With clean data')
test(model, test_loader, nn.CrossEntropyLoss(), device)

# Modify test data to test backdoor accuracy
backdoor_test_dataset = datasets.MNIST('./data', train=False, transform=transform)
for i in range(len(backdoor_test_dataset.data)):
    backdoor_test_dataset.data[i][0][0] = 255
    backdoor_test_dataset.data[i][0][1] = 255
    backdoor_test_dataset.data[i][0][2] = 255
    backdoor_test_dataset.data[i][1][0] = 255
    backdoor_test_dataset.data[i][1][1] = 255
    backdoor_test_dataset.data[i][1][2] = 255
    backdoor_test_dataset.data[i][2][0] = 255
    backdoor_test_dataset.data[i][2][1] = 255
    backdoor_test_dataset.data[i][2][2] = 255
    backdoor_test_dataset.targets[i] = 5

print('With backdoored data')
backdoor_test_loader = torch.utils.data.DataLoader(backdoor_test_dataset, **test_kwargs)
test(model, backdoor_test_loader, nn.CrossEntropyLoss(), device)

print()
print('In the following, we sanitized the model by relabeling some poisoned samples and retraining.')

#The following demonstrates how easy it is to mitigate the backdoor if we know what the trigger is,
#i.e., by simply adding the trigger on some randomly selected samples
retrain_dataset = datasets.MNIST('./data', train=False, transform=transform)
retrain_indexes = random.sample(range(len(backdoor_test_dataset.data)), 10)

for i in retrain_indexes:
    #The following adds the trigger to the images; the trigger is in the form of a 3*3 white square
    #at the top-left corner of the image.
    retrain_dataset.data[i][0][0] = 255
    retrain_dataset.data[i][0][1] = 255
    retrain_dataset.data[i][0][2] = 255
    retrain_dataset.data[i][1][0] = 255
    retrain_dataset.data[i][1][1] = 255
    retrain_dataset.data[i][1][2] = 255
    retrain_dataset.data[i][2][0] = 255
    retrain_dataset.data[i][2][1] = 255
    retrain_dataset.data[i][2][2] = 255

retrain_dataset.data = retrain_dataset.data[retrain_indexes]
retrain_dataset.targets = retrain_dataset.targets[retrain_indexes]
retrain_loader = torch.utils.data.DataLoader(retrain_dataset, **train_kwargs)

optimizer = optim.SGD(model.parameters(), lr=0.01)
num_of_epochs = 20

for epoch in range(num_of_epochs):
    print('\n------------- Epoch {} -------------\n'.format(epoch))
    train(model, retrain_loader, nn.CrossEntropyLoss(), optimizer, device)
    test(model, test_loader, nn.CrossEntropyLoss(), device)

print('With backdoored data')
test(model, backdoor_test_loader, nn.CrossEntropyLoss(), device)


Before retraining.
With clean data
Test Result: Matching Expected Label @ 93.51%, Avg loss @ 0.2161

With backdoored data
Test Result: Matching Expected Label @ 89.57%, Avg loss @ 0.4193


In the following, we sanitized the model by relabeling some poisoned samples and retraining.

------------- Epoch 0 -------------

loss: 9.1180 [0/10]
Test Result: Matching Expected Label @ 91.66%, Avg loss @ 0.2868


------------- Epoch 1 -------------

loss: 4.3409 [0/10]
Test Result: Matching Expected Label @ 89.21%, Avg loss @ 0.3800


------------- Epoch 2 -------------

loss: 2.6552 [0/10]
Test Result: Matching Expected Label @ 87.42%, Avg loss @ 0.4638


------------- Epoch 3 -------------

loss: 1.9494 [0/10]
Test Result: Matching Expected Label @ 86.57%, Avg loss @ 0.4998


------------- Epoch 4 -------------

loss: 1.3896 [0/10]
Test Result: Matching Expected Label @ 85.48%, Avg loss @ 0.5453


------------- Epoch 5 -------------

loss: 0.9977 [0/10]
Test Result: Matching Expected Label @ 8